# 02. 특성 엔지니어링

전처리가 끝난 데이터를 모델이 학습하기 좋은 형태로 바꾸는 방법을 정리합니다.

### 이번 노트에서 배우는 것
- 데이터 분포와 상관관계 확인
- 범주형 데이터 인코딩
- 수치형 데이터 스케일링
- 기존 컬럼을 이용한 파생변수 생성
- 클래스 불균형 확인과 SMOTE
- Train / Validation / Test 분리

> **중요:** SMOTE 같은 오버샘플링은 전체 데이터에 먼저 적용하지 않고, 데이터 분리 후 **훈련 데이터에만 적용**합니다.


In [ ]:
from common.libraries import *

## 1. 데이터 분포 확인

전처리가 끝난 데이터를 불러오고 수치형·범주형 특성의 분포를 간단히 확인합니다.
분포 확인은 이상한 값이 남아 있는지, 특정 범주에 데이터가 지나치게 몰려 있는지 파악하는 데 도움이 됩니다.


In [ ]:
df = pd.read_csv("./data/ecommerce_customers_clean.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Age"], bins=20, kde=True, ax=axes[0])
sns.histplot(df["Tenure"], bins=20, kde=True, ax=axes[1])
axes[0].set_title("Age 분포")
axes[1].set_title("Tenure 분포")
plt.tight_layout()
plt.show()

display(df["Gender"].value_counts())

## 2. 상관관계와 p-value

`corr()`은 수치형 변수 사이의 선형적인 관계를 빠르게 확인하는 방법입니다.

- 상관계수는 `-1 ~ 1`
- 절댓값이 클수록 선형 관계가 강함
- p-value가 작다고 해서 반드시 모델에 좋은 특성이라는 뜻은 아니며, **통계적으로 우연일 가능성이 낮다는 참고 지표**로 사용합니다.


In [ ]:
numeric_df = df.select_dtypes(include="number")
corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("수치형 특성 상관관계")
plt.show()

pearson_coef, pearson_p = pearsonr(df["Tenure"], df["TotalCharges"])
spearman_coef, spearman_p = spearmanr(df["ReviewScore"], df["MonthlyCharges"])

print(f"Tenure ↔ TotalCharges | Pearson: {pearson_coef:.3f}, p-value: {pearson_p:.4f}")
print(f"ReviewScore ↔ MonthlyCharges | Spearman: {spearman_coef:.3f}, p-value: {spearman_p:.4f}")


## 3. 범주형 데이터 인코딩

머신러닝 모델은 문자열을 그대로 계산할 수 없으므로 숫자로 변환해야 합니다.

- **Label Encoding**: 각 범주를 하나의 숫자로 변환
- **One-Hot Encoding**: 범주별 컬럼을 만들어 0/1로 표현

`Gender`처럼 범주가 두 개인 경우 Label Encoding을 간단히 사용할 수 있고,
서열이 없는 여러 범주에는 One-Hot Encoding이 이해하기 쉽습니다.


In [ ]:
df_encoded = df.copy()

label_encoder = LabelEncoder()
df_encoded["Gender_Label"] = label_encoder.fit_transform(df_encoded["Gender"])

# One-Hot Encoding 예시를 위한 임시 범주형 컬럼
np.random.seed(42)
df_encoded["Membership"] = np.random.choice(
    ["Basic", "Silver", "Gold"],
    size=len(df_encoded)
)
df_encoded = pd.get_dummies(df_encoded, columns=["Membership"], dtype=int)

display(df_encoded.head())
print("Gender 인코딩 기준:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


## 4. 수치형 데이터 스케일링

특성마다 단위와 범위가 크게 다르면 거리 기반 모델 등에서 특정 특성의 영향이 과도하게 커질 수 있습니다.

- `StandardScaler` : 평균 0, 표준편차 1
- `MinMaxScaler` : 보통 0~1 범위
- `RobustScaler` : 중앙값과 IQR을 사용해 이상치 영향 완화

여기서는 세 방법의 변환 결과를 비교합니다.


In [ ]:
scale_cols = ["Age", "Tenure", "MonthlyCharges", "TotalCharges"]

scalers = {
    "std": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler(),
}

scaled_preview = pd.DataFrame(index=df_encoded.index)

for name, scaler in scalers.items():
    values = scaler.fit_transform(df_encoded[scale_cols])
    scaled_preview[[f"{col}_{name}" for col in scale_cols]] = values

display(scaled_preview.head())


## 5. 파생변수 만들기

기존 컬럼을 조합해 문제를 더 잘 설명할 수 있는 새로운 특성을 만들 수 있습니다.

- `AvgPaymentPerMonth` : 총 결제액 ÷ 가입 기간
- `Age_Group` : 나이를 구간으로 나눈 연령대

파생변수는 무조건 많이 만드는 것이 아니라 **문제와 데이터 의미에 맞게** 만들어야 합니다.


In [ ]:
df_encoded["AvgPaymentPerMonth"] = (
    df_encoded["TotalCharges"] / df_encoded["Tenure"].replace(0, np.nan)
).fillna(0)

bins = [0, 19, 29, 39, 49, 59, 100]
labels = ["10s", "20s", "30s", "40s", "50s", "60s+"]
df_encoded["Age_Group"] = pd.cut(df_encoded["Age"], bins=bins, labels=labels)
df_encoded["Age_Group_Code"] = df_encoded["Age_Group"].cat.codes

display(df_encoded[["Age", "Age_Group", "Age_Group_Code", "AvgPaymentPerMonth"]].head())


## 6. 클래스 불균형 확인

분류 문제에서는 정답 클래스 비율도 확인합니다.
한 클래스가 지나치게 적으면 모델이 많은 클래스만 잘 맞추는 방향으로 학습할 수 있습니다.

다만 **비율이 다르다고 무조건 SMOTE를 적용하는 것은 아닙니다.**
정확도뿐 아니라 정밀도·재현율·F1 등을 함께 보고 필요할 때 사용합니다.


In [ ]:
class_counts = df_encoded["Churn"].value_counts().sort_index()

display(class_counts.rename("개수"))
display((class_counts / class_counts.sum() * 100).round(2).rename("비율(%)"))

class_counts.plot(kind="bar", title="Churn 클래스 분포")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.show()


## 7. Train / Validation / Test 분리

모델을 공정하게 평가하려면 데이터를 역할별로 나눕니다.

- **Train** : 모델 학습
- **Validation** : 모델/하이퍼파라미터 비교
- **Test** : 마지막 최종 평가

여기서는 6:2:2로 나눕니다.


In [ ]:
feature_cols = [
    "Age", "Tenure", "MonthlyCharges", "TotalCharges",
    "SupportCalls", "ReviewScore", "Gender_Label",
    "AvgPaymentPerMonth", "Age_Group_Code",
]

X = df_encoded[feature_cols]
y = df_encoded["Churn"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    stratify=y,
    random_state=42,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


## 8. SMOTE는 훈련 데이터에만 적용

데이터를 먼저 나눈 뒤 `X_train`, `y_train`에만 SMOTE를 적용합니다.

검증/테스트 데이터까지 합성하면 모델 평가에 사용될 데이터에 훈련 정보가 섞여
실제보다 성능이 좋아 보이는 **데이터 누수(Data Leakage)**가 발생할 수 있습니다.


In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("SMOTE 적용 전")
display(y_train.value_counts().sort_index())

print("SMOTE 적용 후")
display(y_train_balanced.value_counts().sort_index())


### 정리
- 인코딩은 문자열 범주를 모델이 계산할 수 있는 숫자 형태로 바꿉니다.
- 스케일링은 특성의 단위 차이를 줄입니다.
- 파생변수는 기존 데이터를 바탕으로 새로운 정보를 추가합니다.
- 데이터는 **Train / Validation / Test로 먼저 분리**합니다.
- SMOTE 같은 오버샘플링은 **훈련 데이터에만** 적용합니다.
